# Day 10 · Deep Vision Architectures & ResNet

**Objective:** Implement a lightweight ResNet block with an identity shortcut and plug it into the Day 9 healthy-vs-diseased leaf classifier.

## 1. Download the dataset

We use the same Kaggle leaf dataset and binary labels as Day 9. The focus today is the residual architecture.

In [ ]:
!pip -q install kaggle
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!mkdir -p /content/plant_data
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/plant_data --unzip

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_dirs = list(Path('/content/plant_data').rglob('train'))
valid_dirs = list(Path('/content/plant_data').rglob('valid'))
if not train_dirs:
    raise FileNotFoundError("No 'train' folder found after extracting the Kaggle dataset.")
train_dir = train_dirs[0]
valid_dir = valid_dirs[0] if valid_dirs else None
print('Train folder:', train_dir)
print('Validation folder:', valid_dir)

transform = transforms.Compose([transforms.Resize((128, 128)), transforms.ToTensor()])
base_train = datasets.ImageFolder(train_dir, transform=transform)

class BinaryLeafDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, index):
        image, old_label = self.data[index]
        label = 0 if 'healthy' in self.data.classes[old_label].lower() else 1
        return image, torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(BinaryLeafDataset(base_train), batch_size=32, shuffle=True)

## 2. Implement the residual block

The block adds its input to the output of two convolution layers:

\[
y = F(x) + x
\]

If channel dimensions differ, a 1×1 convolution projects the shortcut to the required shape.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = (
            nn.Identity() if in_channels == out_channels else
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

## 3. Plug ResNet block into the classifier

The stem extracts initial features, residual blocks learn deeper features, and global average pooling reduces each feature map to one value before classification.

In [ ]:
class LeafResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        self.residual1 = ResidualBlock(16, 16)
        self.downsample = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(16, 32))
        self.residual2 = ResidualBlock(32, 32)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        x = self.stem(x)
        x = self.residual1(x)
        x = self.downsample(x)
        x = self.residual2(x)
        x = self.pool(x)
        return self.fc(torch.flatten(x, 1))

model = LeafResNet().to(device)
print(model)

## 4. Train the residual classifier

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    print(f'Epoch {epoch+1}/5 | Loss: {total_loss/total:.4f} | Accuracy: {correct/total:.4f}')

### Day 10 · Key takeaway

The identity shortcut adds the block input to its learned transformation, creating a more direct path for information and gradients through the network.